In [1]:
from neo4j import GraphDatabase
import networkx as nx
from getpass import getpass

In [2]:
uri = "bolt://localhost:7687"
username = "neo4j"
password = getpass("Enter Neo4j Password: ")

driver = GraphDatabase.driver(uri, auth=(username, password))

driver.verify_connectivity()

print("Successfully connected to Neo4j!")

Successfully connected to Neo4j!


In [3]:
query = """
MATCH (a:Junction)-[r:ROAD]->(b:Junction)
RETURN a.name AS source,
       b.name AS destination,
       r.distance AS distance,
       r.status AS status,
       r.incident AS incident
"""

with driver.session(database="neo4j") as session:
    result = session.run(query)
    roads = [dict(record) for record in result]

for road in roads:
    print(road)

{'source': 'A', 'destination': 'B', 'distance': 4, 'status': 'Open', 'incident': None}
{'source': 'A', 'destination': 'C', 'distance': 3, 'status': 'Open', 'incident': None}
{'source': 'B', 'destination': 'D', 'distance': 5, 'status': 'Open', 'incident': None}
{'source': 'B', 'destination': 'E', 'distance': 2, 'status': 'Open', 'incident': None}
{'source': 'C', 'destination': 'E', 'distance': 4, 'status': 'Open', 'incident': None}
{'source': 'D', 'destination': 'F', 'distance': 3, 'status': 'Open', 'incident': None}
{'source': 'C', 'destination': 'G', 'distance': 6, 'status': 'Blocked', 'incident': 'Accident'}
{'source': 'F', 'destination': 'G', 'distance': 2, 'status': 'Open', 'incident': None}


In [4]:
original_graph = nx.Graph()
available_graph = nx.Graph()

for road in roads:

    a = road["source"]
    b = road["destination"]
    distance = road["distance"]

    original_graph.add_edge(a, b, weight=distance)
    available_graph.add_nodes_from([a, b])

    if road["status"] == "Open":
        available_graph.add_edge(a, b, weight=distance)

print("Original roads:", original_graph.number_of_edges())
print("Available roads:", available_graph.number_of_edges())

Original roads: 8
Available roads: 7


In [5]:
source = "A"
destination = "F"

original_path = nx.shortest_path(
    original_graph, source, destination, weight="weight"
)

original_distance = nx.shortest_path_length(
    original_graph, source, destination, weight="weight"
)

new_path = nx.shortest_path(
    available_graph, source, destination, weight="weight"
)

new_distance = nx.shortest_path_length(
    available_graph, source, destination, weight="weight"
)

print("Original Route:", " -> ".join(original_path))
print("Original Distance:", original_distance, "km")

print("\nAvailable Route:", " -> ".join(new_path))
print("Available Distance:", new_distance, "km")

Original Route: A -> C -> G -> F
Original Distance: 11 km

Available Route: A -> B -> D -> F
Available Distance: 12 km


In [6]:
blocked_roads = [road for road in roads if road["status"] == "Blocked"]

print("TRANSPORTATION KNOWLEDGE FRAMEWORK")
print("----------------------------------")

for road in blocked_roads:
    print(
        f"Incident: {road['incident'] or 'Unknown'} "
        f"on road {road['source']}-{road['destination']}"
    )

print("\nOriginal Route:", " -> ".join(original_path))
print("Original Distance:", original_distance, "km")

print("\nRecommended Route:", " -> ".join(new_path))
print("Recommended Distance:", new_distance, "km")

if original_path != new_path:
    affected = []

    for road in blocked_roads:
        a, b = road["source"], road["destination"]

        for x, y in zip(original_path, original_path[1:]):
            if {a, b} == {x, y}:
                affected.append(road)

    print("\nEXPLANATION:")

    for road in affected:
        print(
            f"Road {road['source']}-{road['destination']} "
            f"is blocked due to {road['incident'] or 'an incident'}."
        )

    print("The original shortest route is unavailable.")
    print("An alternative route has been calculated.")
    print("Additional Distance:", new_distance - original_distance, "km")
else:
    print("\nEXPLANATION:")
    print("The original shortest route is still available.")

TRANSPORTATION KNOWLEDGE FRAMEWORK
----------------------------------
Incident: Accident on road C-G

Original Route: A -> C -> G -> F
Original Distance: 11 km

Recommended Route: A -> B -> D -> F
Recommended Distance: 12 km

EXPLANATION:
Road C-G is blocked due to Accident.
The original shortest route is unavailable.
An alternative route has been calculated.
Additional Distance: 1 km


In [7]:
driver.close()

print("Neo4j connection closed successfully.")

Neo4j connection closed successfully.
